<a href="https://colab.research.google.com/github/PRR-aiexp/CVYoloMLops/blob/main/CVYoloMLops.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!git clone https://github.com/PRR-aiexp/CVYoloMlops.git
#cd CVYoloMlops

Cloning into 'CVYoloMlops'...
remote: Enumerating objects: 799, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 799 (delta 4), reused 10 (delta 2), pack-reused 783 (from 2)
Receiving objects: 100% (799/799), 37.51 MiB | 31.31 MiB/s, done.
Resolving deltas: 100% (6/6), done.


In [6]:
!pip install ultralytics opencv-python notebook transformers mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.9/76.9 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.9/753.9 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 19.1 MB/s eta 0:00:00


In [7]:
from ultralytics import YOLO
import mlflow
import yaml
import os

# End any active MLflow runs before starting a new one
if mlflow.active_run():
    mlflow.end_run()

# Update the path to the dataset YAML file to reflect the cloned repository structure
DATASET_YAML = "CVYoloMlops/data/yolo_dataset/car_detection.yaml"

# Define the absolute paths for train and validation images and labels
repo_root = "/content/CVYoloMlops/"
yolo_dataset_root = os.path.join(repo_root, "data", "yolo_dataset")

train_images_path = os.path.join(yolo_dataset_root, "images", "train")
val_images_path = os.path.join(yolo_dataset_root, "images", "val")
train_labels_path = os.path.join(yolo_dataset_root, "labels", "train")
val_labels_path = os.path.join(yolo_dataset_root, "labels", "val")

# Create dummy directories to satisfy Ultralytics' initial checks
# In a real scenario, you would have your actual dataset files here.
os.makedirs(train_images_path, exist_ok=True)
os.makedirs(val_images_path, exist_ok=True)
os.makedirs(train_labels_path, exist_ok=True)
os.makedirs(val_labels_path, exist_ok=True)

# New content for car_detection.yaml with absolute paths for robustness
dataset_config = {
    'train': train_images_path,
    'val': val_images_path,
    'nc': 1, # number of classes, assuming 'car' is the only class
    'names': ['car']
}

# Write the updated dataset configuration to the YAML file in the cloned repo
with open(DATASET_YAML, 'w') as f:
    yaml.dump(dataset_config, f)

# Set MLflow experiment name explicitly
mlflow.set_experiment("YOLOv8 Car Detection Experiment")

with mlflow.start_run(run_name="yolov8n_colab_run1"):

    # Log parameters
    mlflow.log_param("model", "yolov8n")
    mlflow.log_param("imgsz", 640)
    mlflow.log_param("epochs", 5)   # start small
    mlflow.log_param("batch", 8)

    model = YOLO("yolov8n.pt")

    # Define project and name for the training run
    train_project = "runs/train"
    train_name = "car_yolo_dagshub_test"

    results = model.train(
        data=DATASET_YAML,
        imgsz=640,
        epochs=5,        # small test first
        batch=8,
        device=0,        # GPU; use "cpu" if no GPU
        project=train_project,
        name=train_name,
        exist_ok=True
    )

    # Log YOLO metrics
    metrics = results.results_dict
    for k, v in metrics.items():
        try:
            mlflow.log_metric(k, float(v))
        except Exception:
            pass

    # Log best model weights as artifact
    # The best model path is constructed from the project and name of the training run
    best_model_path = os.path.join("/content", train_project, train_name, "weights", "best.pt")
    mlflow.log_artifact(best_model_path)

print("Done; run should now be in DagsHub MLflow UI.")

2025/12/09 03:39:22 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/12/09 03:39:22 INFO mlflow.store.db.utils: Updating database tables
2025/12/09 03:39:22 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/09 03:39:22 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2025/12/09 03:39:22 INFO alembic.runtime.migration: Running upgrade  -> 451aebb31d03, add metric step
2025/12/09 03:39:22 INFO alembic.runtime.migration: Running upgrade 451aebb31d03 -> 90e64c465722, migrate user column to tags
2025/12/09 03:39:22 INFO alembic.runtime.migration: Running upgrade 90e64c465722 -> 181f10493468, allow nulls for metric values
2025/12/09 03:39:22 INFO alembic.runtime.migration: Running upgrade 181f10493468 -> df50e92ffc5e, Add Experiment Tags Table
2025/12/09 03:39:22 INFO alembic.runtime.migration: Running upgrade df50e92ffc5e -> 7ac759974ad8, Update run tags with larger limit
2025/12/09 03:39:22 INFO alembic.runtime.migration: Running 

Ultralytics 8.3.235 🚀 Python-3.12.12 torch-2.9.0+cu126 


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 0
os.environ['CUDA_VISIBLE_DEVICES']: None
See https://pytorch.org/get-started/locally/ for up-to-date torch install instructions if no CUDA devices are seen by torch.
